# Notebook clean pour les étapes d'Extract & Transform

## 1 - EXTRACT

In [1]:
# Charger les dépendances

import os
import yaml
import pandas as pd
import numpy as np
import requests
from sqlalchemy import create_engine
from dotenv import load_dotenv
from pathlib import Path
import sys
from dataclasses import dataclass
from pathlib import Path

In [2]:
load_dotenv()

True

In [3]:
try:
    ROOT_DIR = Path(__file__).resolve().parents[1]
except NameError:
    ROOT_DIR = Path.cwd().parent

CONFIG_PATH = ROOT_DIR / "config.yml"
print(CONFIG_PATH)

/Users/LounesAbd/Etudes/Simplon_DataEng/Projects/1_trafic-road-accidents/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/config.yml


In [4]:
def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
        
    for section, values in config.items():
        for key, val in values.items():
            if isinstance(val, str) and val.startswith("${"):
                env_var = val.strip("${}")
                config[section][key] = os.getenv(env_var)    
    return config

In [5]:
csv = ROOT_DIR / "data" / "acc_2017.csv"
print(csv)

/Users/LounesAbd/Etudes/Simplon_DataEng/Projects/1_trafic-road-accidents/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/acc_2017.csv


In [6]:
"""Téléchargement minimaliste du dataset accidents corporels depuis OpenDataSoft.

Version simplifiée sans retry, sans barre de progression, sans validation.
Télécharge le CSV par chunks et le sauvegarde dans data/accidents_corporels_millesime.csv

Usage:
    python scripts/sauvegarde_csv_api_v1.py

Source:
    https://public.opendatasoft.com - Dataset accidents corporels de la circulation
"""


@dataclass(frozen=True)
class Config:
    """Configuration du téléchargement."""
    
    base_url: str = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets"
    dataset_id: str = "accidents-corporels-de-la-circulation-millesime"
    output_dir: str = "data"
    output_file: str = "accidents_corporels_millesime.csv"
    delimiter: str = ","
    chunk_size: int = 65536 #on lit 64KB par 64KB pour ne pas que Lounes voit la RAM de son pc bruler
    timeout: int = 30


def build_url(config: Config) -> str:
    """Construit l'URL de téléchargement."""
    return f"{config.base_url}/{config.dataset_id}/exports/csv?delimiter={config.delimiter}"


def resolve_path(config: Config) -> Path:
    """Détermine le chemin de sortie."""
    #script_dir = ROOT_DIR
    return ROOT_DIR / config.output_dir / config.output_file


def download_csv(url: str, destination: Path, config: Config) -> None:
    """Télécharge le CSV par chunks."""
    print(f"Téléchargement depuis OpenDataSoft...")

    response = requests.get(url, stream=True, timeout=config.timeout)
    response.raise_for_status()

    destination.parent.mkdir(parents=True, exist_ok=True)

    with response, open(destination, "wb") as handle:
        for chunk in response.iter_content(chunk_size=config.chunk_size):
            if chunk:
                handle.write(chunk)

    print(f"Fichier sauvegardé: {destination}")


def main() -> None:
    """Point d'entrée principal."""
    config = Config()
    url = build_url(config)
    path = resolve_path(config)

    download_csv(url, path, config)

    print("Téléchargement terminé")


if __name__ == "__main__":
    main()


Téléchargement depuis OpenDataSoft...
Fichier sauvegardé: /Users/LounesAbd/Etudes/Simplon_DataEng/Projects/1_trafic-road-accidents/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
Téléchargement terminé


In [7]:
# Importation des données source dans un DataFrame pandas

df_source = pd.read_csv(ROOT_DIR / 'data/accidents_corporels_millesime.csv', delimiter=',', low_memory=False)
df_source.head()

,num_acc,datetime,nom_com,an,mois,jour,hrmn,lum,agg,int,...,year_georef,com_name,dep_code,dep_name,epci_code,epci_name,reg_code,reg_name,com_arm_name,com_code
0,201900020750,2019-01-29T15:45:00+00:00,Corbeil-essonnes,2019,1,29,16:45,Plein jour,En agglomération,2,...,2019,Corbeil-Essonnes,91.0,Essonne,200059228.0,CA Grand Paris Sud Seine Essonne Sénart,11.0,Île-de-France,Corbeil-Essonnes,91174.0
1,201900020796,2019-10-07T17:30:00+00:00,Istres,2019,10,7,19:30,Nuit sans éclairage public,Hors agglomération,1,...,2019,Istres,13.0,Bouches-du-Rhône,200054807.0,Métropole d'Aix-Marseille-Provence,93.0,Provence-Alpes-Côte d'Azur,Istres,13047.0
2,201900020869,2019-10-13T13:46:00+00:00,Saint-laurent-du-pont,2019,10,13,15:46,Plein jour,En agglomération,1,...,2019,Saint-Laurent-du-Pont,38.0,Isère,200040111.0,CC Coeur de Chartreuse,84.0,Auvergne-Rhône-Alpes,Saint-Laurent-du-Pont,38412.0
3,201900021309,2019-04-17T14:30:00+00:00,Livry-gargan,2019,4,17,16:30,Plein jour,En agglomération,1,...,2019,Livry-Gargan,93.0,Seine-Saint-Denis,200054781.0,Métropole du Grand Paris,11.0,Île-de-France,Livry-Gargan,93046.0
4,201900018753,2019-12-25T17:10:00+00:00,Gennevilliers,2019,12,25,18:10,Nuit sans éclairage public,Hors agglomération,1,...,2019,Gennevilliers,92.0,Hauts-de-Seine,200054781.0,Métropole du Grand Paris,11.0,Île-de-France,Gennevilliers,92036.0


In [8]:
df_source.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 69 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   num_acc       475911 non-null  int64  
 1   datetime      475911 non-null  object 
 2   nom_com       449206 non-null  object 
 3   an            475911 non-null  int64  
 4   mois          475911 non-null  int64  
 5   jour          475911 non-null  int64  
 6   hrmn          475911 non-null  object 
 7   lum           475911 non-null  object 
 8   agg           475911 non-null  object 
 9   int           475911 non-null  int64  
 10  atm           475861 non-null  object 
 11  col           475901 non-null  object 
 12  dep           475911 non-null  object 
 13  com           475911 non-null  object 
 14  insee         475296 non-null  float64
 15  adr           426655 non-null  object 
 16  lat           300785 non-null  object 
 17  long          300785 non-null  object 
 18  code

## 2 - TRANSFORM

In [13]:
def preprocess_and_split(df_source):
    """
    Clean the source dataframe and split it into 5 separate dataframes:
    accidents, lieux, date_accident, vehicules, usagers.
    
    Parameters:
        df_source (pd.DataFrame): Raw input dataframe.
        
    Returns:
        dict: A dictionary containing the 5 dataframes.
    """
    
    # Tell Python these variables are global
    global df_accidents, df_lieux, df_date_accident, df_vehicules, df_usagers

    # Step 1: Log starting
    print("Step 1: Starting preprocessing...")

    # Clean column names
    df_source.columns = df_source.columns.str.lower().str.strip()
    print("Columns lowercased and stripped.")
    
    # Convert datetime
    df_source["datetime"] = pd.to_datetime(df_source["datetime"], errors="coerce")
    print("Datetime column converted.")
    
    # Step 2: Create 5 separate dataframes
    print("Step 2: Splitting into 5 source dataframes...")

    df_accidents = df_source[[
        'num_acc', 'lum', 'agg', 'int', 'atm', 'adr', 'col', 'circ', 'plan', 
        'prof', 'surf', 'infra', 'situ', 'year_georef'
    ]].copy()
    print("Accidents dataframe created:", df_accidents.shape)

    df_lieux = df_source[[
        'com_code', 'com_name', 'dep_code', 'dep_name', 'reg_code', 'reg_name', 
        'epci_code', 'epci_name', 'lat', 'long', 'catr', 'v1', 'voie', 'v2', 
        'nbv', 'vosp', 'pr', 'pr1', 'lartpc', 'larrout', 'num_acc'
    ]].copy()
    print("Lieux dataframe created:", df_lieux.shape)

    df_date_accident = df_source[[
        'datetime', 'an', 'mois', 'jour', 'hrmn', 'num_acc'
    ]].copy()
    print("Date_accident dataframe created:", df_date_accident.shape)

    df_vehicules = df_source[[
        'num_veh', 'catv', 'choc', 'senc', 'obs', 'obsm', 'occutc', 'manv', 'num_acc'
    ]].copy()
    print("Vehicules dataframe created:", df_vehicules.shape)

    df_usagers = df_source[[
        'sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 
        'locp', 'actp', 'etatp', 'num_acc'
    ]].copy()
    print("Usagers dataframe created:", df_usagers.shape)
    
    print("Step 3: All 5 dataframes created and ready for transformation.")
    
    # Return just a simple message instead of full dataframes
    return "✅ Preprocessing done. 5 dataframes are created and ready for transformation."

In [14]:
preprocess_and_split(df_source)

Step 1: Starting preprocessing...
Columns lowercased and stripped.
Datetime column converted.
Step 2: Splitting into 5 source dataframes...


Accidents dataframe created: (475911, 14)
Lieux dataframe created: (475911, 21)
Date_accident dataframe created: (475911, 6)
Vehicules dataframe created: (475911, 9)
Usagers dataframe created: (475911, 11)
Step 3: All 5 dataframes created and ready for transformation.


'✅ Preprocessing done. 5 dataframes are created and ready for transformation.'

In [16]:
def transform_accidents():
    """
    Transform df_accidents and create df_accidents_cleaned.
    - Replace '-1' with NaN for categorical columns
    - Clean 'infra' and 'situ' columns
    Logs each step.
    """
    
    global df_accidents, df_accidents_cleaned
    
    print("🔹 Starting transformation on df_accidents...")
    
    # Copy the original dataframe
    df_accidents_cleaned = df_accidents.copy()
    
    cat_cols = ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']
    
    # Step 1: Replace '-1' with NaN
    for col in cat_cols:
        df_accidents_cleaned[col] = df_accidents_cleaned[col].replace('-1', np.nan)
    print(f"Step 1: Replaced '-1' with NaN for columns: {cat_cols}")
    
    # Step 2: Clean 'infra' and 'situ' columns
    df_accidents_cleaned['infra'] = df_accidents_cleaned['infra'].apply(lambda x: np.nan if str(x).isdigit() else x)
    df_accidents_cleaned['situ'] = df_accidents_cleaned['situ'].apply(lambda x: np.nan if str(x).isdigit() else x)
    print("Step 2: Cleaned 'infra' and 'situ' columns (numbers converted to NaN if present)")
   
    print("✅ df_accidents_cleaned transformation completed.\n")
    print(df_accidents_cleaned.info())

In [17]:
transform_accidents()

🔹 Starting transformation on df_accidents...
Step 1: Replaced '-1' with NaN for columns: ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']
Step 2: Cleaned 'infra' and 'situ' columns (numbers converted to NaN if present)
✅ df_accidents_cleaned transformation completed.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   num_acc      475911 non-null  int64 
 1   lum          475911 non-null  object
 2   agg          475911 non-null  object
 3   int          475911 non-null  int64 
 4   atm          475860 non-null  object
 5   adr          426655 non-null  object
 6   col          475899 non-null  object
 7   circ         447350 non-null  object
 8   plan         441476 non-null  object
 9   prof         447000 non-null  object
 10  surf         459759 non-null  object
 11  infra        51623 non-null   obj

In [18]:
def transform_lieux():
    """
    Transform df_lieux and create df_lieux_cleaned.
    - Drop unnecessary columns
    - Clean 'catr', 'nbv', and 'vosp' columns
    Logs each step.
    """
    
    global df_lieux, df_lieux_cleaned
    
    print("🔹 Starting transformation on df_lieux...")
    
    # Copy original dataframe
    df_lieux_cleaned = df_lieux.copy()
    
    # Step 1: Drop unnecessary columns
    drop_cols = ['voie', 'v1', 'v2', 'pr', 'pr1', 'lartpc', 'larrout', 
                 'epci_code', 'epci_name', 'lat', 'long']
    df_lieux_cleaned = df_lieux_cleaned.drop(columns=drop_cols)
    print(f"Step 1: Dropped columns: {drop_cols}")
    
    # Step 2: Clean 'catr'
    df_lieux_cleaned['catr'] = df_lieux_cleaned['catr'].apply(lambda x: 'autre' if str(x).isdigit() else x)
    print("Step 2: Cleaned 'catr' column (numbers converted to 'autre')")
    
    # Step 3: Clean 'nbv'
    df_lieux_cleaned['nbv'] = df_lieux_cleaned['nbv'].apply(
        lambda x: np.nan if (x > 6.0 or x == 0.0 or x == -1.0) else x
    )
    df_lieux_cleaned['nbv'] = df_lieux_cleaned['nbv'].astype('Int64')
    print("Step 3: Cleaned 'nbv' column and converted dtype to Int64")
    
    # Step 4: Clean 'vosp'
    df_lieux_cleaned['vosp'] = df_lieux_cleaned['vosp'].apply(lambda x: np.nan if x == '-1' else x)
    print("Step 4: Cleaned 'vosp' column ('-1' replaced with NaN)")
    
    print("✅ df_lieux_cleaned transformation completed.\n")
    print(df_lieux_cleaned.info())


In [19]:
transform_lieux()

🔹 Starting transformation on df_lieux...
Step 1: Dropped columns: ['voie', 'v1', 'v2', 'pr', 'pr1', 'lartpc', 'larrout', 'epci_code', 'epci_name', 'lat', 'long']
Step 2: Cleaned 'catr' column (numbers converted to 'autre')
Step 3: Cleaned 'nbv' column and converted dtype to Int64
Step 4: Cleaned 'vosp' column ('-1' replaced with NaN)
✅ df_lieux_cleaned transformation completed.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 10 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   com_code  475296 non-null  float64
 1   com_name  464648 non-null  object 
 2   dep_code  464648 non-null  float64
 3   dep_name  464648 non-null  object 
 4   reg_code  464648 non-null  float64
 5   reg_name  464648 non-null  object 
 6   catr      475911 non-null  object 
 7   nbv       424073 non-null  Int64  
 8   vosp      31411 non-null   object 
 9   num_acc   475911 non-null  int64  
dtypes: Int64(1), float64(3

In [20]:
def transform_date_accident():
    """
    Transform df_date_accident and create df_date_accident_cleaned.
    - Convert 'hrmn' column from string/object to datetime.time
    Logs each step.
    """
    
    global df_date_accident, df_date_accident_cleaned
    
    print("🔹 Starting transformation on df_date_accident...")
    
    # Copy the original dataframe
    df_date_accident_cleaned = df_date_accident.copy()
    
    # Step 1: Convert 'hrmn' to datetime.time
    df_date_accident_cleaned['hrmn'] = pd.to_datetime(
        df_date_accident_cleaned['hrmn'], format='%H:%M', errors='coerce'
    ).dt.time
    print("Step 1: Converted 'hrmn' column to datetime.time")
    
    print("✅ df_date_accident_cleaned transformation completed.\n")
    print(df_date_accident_cleaned.info())


In [21]:
transform_date_accident()

🔹 Starting transformation on df_date_accident...
Step 1: Converted 'hrmn' column to datetime.time
✅ df_date_accident_cleaned transformation completed.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype              
---  ------    --------------   -----              
 0   datetime  475911 non-null  datetime64[ns, UTC]
 1   an        475911 non-null  int64              
 2   mois      475911 non-null  int64              
 3   jour      475911 non-null  int64              
 4   hrmn      475911 non-null  object             
 5   num_acc   475911 non-null  int64              
dtypes: datetime64[ns, UTC](1), int64(4), object(1)
memory usage: 21.8+ MB
None


In [22]:
# FONCTION D'EXPLOSION - VEHICULES ET USAGERS

def explode_df(df, cols_to_explode, id_col_name):
    """
    Explodes a DataFrame where some columns contain comma-separated values.
    Handles list alignment, padding, and adds a sequential unique ID column.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.
    cols_to_explode : list
        List of column names that contain comma-separated or list values.
    id_col_name : str
        Name of the ID column to create (e.g. 'vehicule_id', 'usager_id').
    
    Returns
    -------
    pd.DataFrame
        Exploded DataFrame with a unique sequential ID as the first column.
    """
    df = df.copy()

    # 1️⃣ Ensure all values are lists (split by comma if needed)
    for col in cols_to_explode:
        df[col] = df[col].astype(str).apply(lambda x: x.split(',') if ',' in x else [x])

    # 2️⃣ Compute length consistency per row
    list_lengths_df = df[cols_to_explode].map(lambda x: len(x) if isinstance(x, list) else 1)
    df['nunique_lengths'] = list_lengths_df.nunique(axis=1)
    df['max_length'] = list_lengths_df.max(axis=1)

    # 3️⃣ Detect misaligned rows
    misaligned_rows = df[df['nunique_lengths'] > 1]
    if len(misaligned_rows) > 0:
        print(f"⚠️ {len(misaligned_rows)} rows with misaligned list lengths detected — they will be padded.")

    # 4️⃣ Pad lists to same length
    def pad_lists(row):
        max_len = row['max_length']
        for col in cols_to_explode:
            vals = row[col] if isinstance(row[col], list) else [row[col]]
            row[col] = (vals + [None] * (max_len - len(vals)))[:max_len]
        return row

    df = df.apply(pad_lists, axis=1)

    # 5️⃣ Explode all relevant columns together
    df_exploded = df.explode(cols_to_explode, ignore_index=True)
    print(f"✓ Data exploded successfully: {len(df_exploded):,} rows.")

    # 6️⃣ Add sequential unique ID
    df_exploded = df_exploded.reset_index(drop=True)
    df_exploded[id_col_name] = df_exploded.index + 1

    # 7️⃣ Reorder columns to put ID first
    cols = [id_col_name] + [c for c in df_exploded.columns if c != id_col_name]
    df_exploded = df_exploded[cols]

    return df_exploded

In [23]:
def transform_vehicules():
    """
    Transform df_vehicules and create df_vehicules_cleaned.
    - Explode columns with comma-separated values
    - Clean categorical columns ('catv', 'choc', 'obs', 'obsm', 'occutc', 'manv')
    - Adjust dtypes
    Logs each step.
    """
    
    global df_vehicules, df_vehicules_cleaned
    
    print("🔹 Starting transformation on df_vehicules...")
    
    # Step 0: Copy original df
    df_veh_cleaned = df_vehicules.copy()
    
    # Step 1: Explode relevant columns
    cols_to_explode = ['catv', 'choc', 'obs', 'obsm', 'occutc', 'manv']
    df_veh_exploded = explode_df(df_veh_cleaned, cols_to_explode, id_col_name='vehicule_id')
    print(f"Step 1: Exploded columns: {cols_to_explode}")
    
    # Step 2: Clean 'catv' column
    def clean_catv(value):
        if pd.isna(value):
            return value
        if value == '0':
            return 'Autre véhicule'
        if value in ['50', '43', '42', '41', '60', '80']:
            return 'Autre véhicule'
        if 'Voiturette (Quadricycle à moteur carrossé (anciennement "voiturette ou tricycle à moteur"))' in value:
            return 'Voiturette'
        if 'Quad léger <= 50 cm3 (Quadricycle à moteur non carrossé)' in value:
            return 'Quad léger <= 50 cm3'
        if 'Quad lourd > 50 cm3 (Quadricycle à moteur non carrossé)' in value:
            return 'Quad lourd > 50 cm3'
        return value

    df_veh_exploded['catv'] = df_veh_exploded['catv'].apply(clean_catv)
    print("Step 2: Cleaned 'catv' column")
    
    # Step 3: Clean generic columns
    def clean_generic(value):
        if pd.isna(value) or value in ['nan', 'None']:
            return np.nan
        if value == '-1':
            return np.nan
        return value

    for col in ['choc','obs', 'obsm', 'occutc', 'manv']:
        df_veh_exploded[col] = df_veh_exploded[col].apply(clean_generic)
    print("Step 3: Cleaned 'choc', 'obs', 'obsm', 'occutc', 'manv' columns")
    
    # Step 4: Adjust dtypes
    for cols in ['obs', 'manv']:
        df_veh_exploded[cols] = df_veh_exploded[cols].apply(lambda x: np.nan if str(x).isdigit() else x)
    df_veh_exploded['occutc'] = df_veh_exploded['occutc'].astype('Int64')
    print("Step 4: Adjusted dtypes for 'obs', 'manv', and 'occutc'")
    
    # Step 5: Drop helper columns from explode_df
    df_veh_exploded = df_veh_exploded.drop(columns=['nunique_lengths', 'max_length'])
    print("Step 5: Dropped helper columns from explode_df")
    
    # Step 6: Assign to global cleaned df
    df_vehicules_cleaned = df_veh_exploded
    print("✅ df_vehicules_cleaned transformation completed.\n")
    print(df_vehicules_cleaned.info())


In [24]:
transform_vehicules()

🔹 Starting transformation on df_vehicules...
⚠️ 290537 rows with misaligned list lengths detected — they will be padded.
✓ Data exploded successfully: 811,335 rows.
Step 1: Exploded columns: ['catv', 'choc', 'obs', 'obsm', 'occutc', 'manv']
Step 2: Cleaned 'catv' column
Step 3: Cleaned 'choc', 'obs', 'obsm', 'occutc', 'manv' columns
Step 4: Adjusted dtypes for 'obs', 'manv', and 'occutc'
Step 5: Dropped helper columns from explode_df
✅ df_vehicules_cleaned transformation completed.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 811335 entries, 0 to 811334
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   vehicule_id  811335 non-null  int64 
 1   num_veh      811335 non-null  object
 2   catv         811335 non-null  object
 3   choc         754556 non-null  object
 4   senc         363930 non-null  object
 5   obs          106156 non-null  object
 6   obsm         653089 non-null  object
 7   occutc       6088 

In [25]:
def transform_usagers():
    """
    Transform df_usagers and create df_usagers_cleaned.
    - Explode columns with comma-separated values using explode_df
    - Apply cleaning logic directly
    - Drop helper columns
    Logs each step.
    """
    
    global df_usagers, df_usagers_cleaned
    
    print("🔹 Starting transformation on df_usagers...")
    
    # Step 0: Copy original
    df_us_cleaned = df_usagers.copy()
    
    # Step 1: Explode relevant columns
    cols_to_explode = ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 
                       'catu', 'place', 'locp', 'actp', 'etatp']
    df_usager_exploded = explode_df(df_us_cleaned, cols_to_explode, id_col_name='usager_id')
    print(f"Step 1: Exploded columns: {cols_to_explode}")
    
    # Step 2: Normalize strings and strip whitespace
    str_cols = df_usager_exploded.select_dtypes(include="object").columns
    for col in str_cols:
        df_usager_exploded[col] = df_usager_exploded[col].astype(str).str.strip()
    print("Step 2: Normalized string columns (stripped whitespace)")
    
    # Step 3: Replace string missing values with np.nan
    df_usager_exploded.replace(to_replace=['nan', 'NaN', 'None', '-1'], value=np.nan, inplace=True)
    print("Step 3: Replaced 'nan', 'NaN', 'None', '-1' with np.nan")
    
    # Step 4: Clean 'locp' and 'actp'
    df_usager_exploded['locp'] = df_usager_exploded['locp'].replace(r'^\d+$', np.nan, regex=True)
    df_usager_exploded['actp'] = df_usager_exploded['actp'].replace(r'^\d+$', np.nan, regex=True)
    df_usager_exploded['actp'] = df_usager_exploded['actp'].replace(['A', 'B'], np.nan)
    print("Step 4: Cleaned 'locp' and 'actp' columns")
    
    # Step 5: Nullify pedestrian-only columns where no pedestrian in accident
    mask_pieton_present = df_usager_exploded.groupby("num_acc")["catu"].transform(lambda x: (x == "Piéton").any())
    cols_to_null = ["locp", "actp", "etatp"]
    df_usager_exploded.loc[~mask_pieton_present, cols_to_null] = np.nan
    print("Step 5: Nullified pedestrian-only columns where no pedestrian in accident")
    
    # Step 6: Drop helper columns from explode_df
    df_usager_exploded = df_usager_exploded.drop(columns=['nunique_lengths', 'max_length'])
    print("Step 6: Dropped helper columns from explode_df")
    
    # Step 7: Assign to global cleaned DataFrame
    df_usagers_cleaned = df_usager_exploded
    print("✅ df_usagers_cleaned transformation completed.\n")
    print(df_usagers_cleaned.info())


In [26]:
transform_usagers()

🔹 Starting transformation on df_usagers...
⚠️ 385795 rows with misaligned list lengths detected — they will be padded.
✓ Data exploded successfully: 1,061,254 rows.
Step 1: Exploded columns: ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 'locp', 'actp', 'etatp']
Step 2: Normalized string columns (stripped whitespace)
Step 3: Replaced 'nan', 'NaN', 'None', '-1' with np.nan
Step 4: Cleaned 'locp' and 'actp' columns
Step 5: Nullified pedestrian-only columns where no pedestrian in accident
Step 6: Dropped helper columns from explode_df
✅ df_usagers_cleaned transformation completed.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1061254 entries, 0 to 1061253
Data columns (total 12 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   usager_id  1061254 non-null  int64 
 1   sexe       1061254 non-null  object
 2   grav       1061254 non-null  object
 3   trajet     778342 non-null   object
 4   secu       885685 non-null   obje